# Metrics Table by Dataset and Action

This notebook loads `results/all_models_metrics_long.csv`, filters rows by `dataset` and `action_filter`, and displays a styled table where the best value for each metric is highlighted in green and the second best in amber.

`APD` is treated as higher-is-better. The other displayed metrics are treated as lower-is-better.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

CSV_PATH = Path('/home/fagnelli/diffusion_hands/results/all_models_metrics_long.csv')
METRIC_COLUMNS = [
    'APD',
    'ADE',
    'FDE',
    'MMADE',
    'MMFDE',
    'CMD',
    'FID',
]
HIGHER_IS_BETTER = {'APD'}

df = pd.read_csv(CSV_PATH)
df['action_filter'] = df['action_filter'].fillna('')


BEST_STYLE = 'font-weight: bold; color: #0f5132; background-color: #d1e7dd;'
SECOND_STYLE = 'font-weight: 600; color: #664d03; background-color: #fff3cd;'


def build_metrics_table(dataset, action_filter=''):
    action_filter = '' if action_filter is None else str(action_filter)

    filtered = df.loc[
        (df['status'] == 'ok')
        & (df['dataset'] == dataset)
        & (df['action_filter'] == action_filter),
        ['model', *METRIC_COLUMNS],
    ].copy()

    if filtered.empty:
        raise ValueError(
            f'No rows found for dataset={dataset!r} and action_filter={action_filter!r}.'
        )

    filtered = filtered.sort_values('model').reset_index(drop=True)

    def highlight_best_and_second(series):
        styles = ['' for _ in series]
        ascending = series.name not in HIGHER_IS_BETTER
        ranked = series.dropna().sort_values(ascending=ascending, kind='stable')
        unique_values = ranked.drop_duplicates().tolist()

        if unique_values:
            best_value = unique_values[0]
            for idx in series.index[series.eq(best_value)]:
                styles[idx] = BEST_STYLE

        if len(unique_values) > 1:
            second_value = unique_values[1]
            for idx in series.index[series.eq(second_value)]:
                styles[idx] = (styles[idx] + ' ' + SECOND_STYLE).strip()

        return styles

    styled = (
        filtered.style.format({metric: '{:.3f}' for metric in METRIC_COLUMNS})
        .apply(highlight_best_and_second, subset=METRIC_COLUMNS, axis=0)
    )
    return filtered, styled


In [7]:
dataset_filter = 'assembly'
action_filter = 'all'

if action_filter == 'all':
    available_actions = (
        df.loc[df['dataset'] == dataset_filter, 'action_filter']
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    if not available_actions:
        raise ValueError(f'No rows found for dataset={dataset_filter!r}.')

    for current_action in available_actions:
        table, styled_table = build_metrics_table(dataset_filter, current_action)
        print(f"dataset={dataset_filter!r}, action_filter={current_action!r}, rows={len(table)}")
        display(styled_table)
else:
    table, styled_table = build_metrics_table(dataset_filter, action_filter)
    print(f"dataset={dataset_filter!r}, action_filter={action_filter!r}, rows={len(table)}")
    display(styled_table)


dataset='assembly', action_filter='attempt', rows=7


,model,APD,ADE,FDE,MMADE,MMFDE,CMD,FID
0,belfusion,5.688,1.643,1.767,1.641,1.763,0.041,0.709
1,comusion,9.365,0.719,0.979,0.794,0.999,0.005,0.296
2,dlow_cvae,7.634,0.884,1.102,0.884,1.102,0.018,2.753
3,gsps,10.461,0.778,1.070,0.807,1.069,0.011,0.636
4,humanmac,9.635,0.888,1.138,0.902,1.145,0.010,0.259
5,skeletondiffusion,7.149,1.022,1.305,1.038,1.317,0.012,0.357
6,twostage_dct_diffusion,5.692,0.701,0.956,0.782,0.984,0.010,0.064


dataset='assembly', action_filter='pick', rows=7


,model,APD,ADE,FDE,MMADE,MMFDE,CMD,FID
0,belfusion,6.980,1.793,1.875,1.792,1.874,0.030,0.287
1,comusion,11.384,0.872,1.140,1.016,1.207,0.006,0.306
2,dlow_cvae,8.834,1.001,1.201,1.001,1.201,0.016,3.173
3,gsps,9.611,0.867,1.140,0.897,1.148,0.010,0.120
4,humanmac,11.604,1.101,1.346,1.123,1.355,0.008,0.183
5,skeletondiffusion,10.756,1.269,1.580,1.346,1.666,0.006,1.610
6,twostage_dct_diffusion,6.924,0.826,1.088,1.006,1.200,0.008,0.000


dataset='assembly', action_filter='pick_up_screwd', rows=8


...

,model,APD,ADE,FDE,MMADE,MMFDE,CMD,FID
0,belfusion,6.292,1.679,1.737,1.677,1.735,0.034,0.373
1,comusion,9.438,0.802,1.084,0.885,1.111,0.004,0.000
2,dlow_cvae,8.654,0.999,1.235,0.999,1.235,0.021,3.344
3,gsps,10.145,0.844,1.148,0.868,1.148,0.010,0.226
4,humanmac,9.909,0.965,1.209,0.982,1.212,0.008,0.047
5,skeletondiffusion,8.681,1.320,1.663,1.339,1.687,0.007,1.522
6,twostage_dct_diffusion,6.351,0.783,1.065,0.881,1.106,0.007,0.000


dataset='assembly', action_filter='rotate', rows=7


,model,APD,ADE,FDE,MMADE,MMFDE,CMD,FID
0,belfusion,6.562,1.895,1.951,1.894,1.950,0.046,0.737
1,comusion,13.578,0.967,1.280,1.041,1.301,0.006,0.880
2,dlow_cvae,9.511,1.145,1.356,1.145,1.356,0.023,4.328
3,gsps,9.643,0.948,1.247,0.968,1.249,0.018,0.483
4,humanmac,13.266,1.201,1.503,1.212,1.503,0.011,0.718
5,skeletondiffusion,9.531,1.566,1.971,1.579,1.983,0.014,2.110
6,twostage_dct_diffusion,7.299,0.915,1.218,0.999,1.256,0.013,0.184


dataset='assembly', action_filter='unscrew', rows=7


,model,APD,ADE,FDE,MMADE,MMFDE,CMD,FID
0,belfusion,5.832,1.456,1.473,1.454,1.470,0.039,0.762
1,comusion,9.535,0.752,0.910,0.856,0.946,0.004,0.578
2,dlow_cvae,7.539,0.935,0.983,0.935,0.983,0.027,2.689
3,gsps,10.192,0.794,0.976,0.820,0.976,0.020,0.883
4,humanmac,8.294,0.879,1.035,0.903,1.048,0.018,0.410
5,skeletondiffusion,6.894,1.147,1.343,1.191,1.384,0.019,1.589
6,twostage_dct_diffusion,6.251,0.732,0.873,0.852,0.927,0.008,0.104


In [ ]:
# Optional helper: inspect available filter values.
available_filters = (
    df[['dataset', 'action_filter']]
    .drop_duplicates()
    .sort_values(['dataset', 'action_filter'])
    .reset_index(drop=True)
)
display(available_filters)
